# CENSOR Frobenius-Norm Variations -- Extended Edition

This notebook extracts the CENSOR mechanism from the official repo
(https://github.com/KaiyuanZh/CENSOR, see `defense.py`) and compares **32
Frobenius/orthogonal-projection-style defense variants** against two
final-layer label-inference proxy attacks. The paper's core idea is to
replace the client gradient with a sampled update from a subspace orthogonal
to the original gradient, then select a useful update by a cold-posterior
loss check.

Paper: https://kaiyuanzhang.com/publications/NDSS25_Censor.pdf ("CENSOR:
Defense Against Gradient Inversion via Orthogonal Subspace Bayesian
Sampling", NDSS 2025).

**Read this before running anything -- what a 100% number here does and
does not mean.** This notebook does **not** implement the paper's actual
attacks (IG, GI, GGL, GIAS, GIFD) or its image-reconstruction metrics
(MSE/LPIPS/PSNR/SSIM). It implements a much cheaper proxy: the final-FC-layer
*label*-inference step (paper Eq. 7 / Zhao et al.'s iDLG, 2020) that several
real attacks use internally. `raw_no_defense` leaking the label 100% of the
time is simply a reproduction of the well-known, **pre-CENSOR** fact that an
undefended FC gradient trivially reveals its label -- the exact problem
CENSOR exists to fix, not a new break. `row_wise_final_layer_frobenius`
leaking ~100% is a **deliberate negative control** (see its docstring) that
confirms this harness measures what it claims to. Neither number says
anything about CENSOR's real, paper-reported effectiveness against actual
image-reconstruction attacks (Table I: CENSOR reduces PSNR from ~30 to ~11 dB
and pushes SSIM from ~0.93 to ~0.08 on CIFAR-10 against GIAS, for instance).
**A full mathematical discussion of this point, plus the math behind every
variant below, is in the accompanying PDF research report.**

**What's new in this edition:**
- Two real bug fixes: (1) BatchNorm mode during attack/defense evaluation
  was left in `.train()`; (2) eval samples were non-shuffled; (3) this
  notebook previously called a stale 1-argument `summarize(rows)` against a
  `run_experiment` that already returned `(rows, meta)` -- both are now
  consistent (`rows, meta = run_experiment(args)` / `summarize(rows, meta)`).
- `repo_layerwise_as_is` now reproduces the *exact* bug present in the
  official `defense.py` (confirmed by reading it directly from GitHub),
  rather than an invented approximation of it.
- **21 new defense variants** across four mathematical axes: (A) the
  *distribution/structure* of the random candidate g_r (spectral-subspace,
  DCT-domain, Johnson-Lindenstrauss, Rademacher, Laplace); (B) *what
  second-order statistic gets normalized/preserved* (spectral-norm matching,
  row-norm equalization, row-norm permutation, Dirichlet mass reallocation,
  bias-only vs weight-only protection); (C) the *cold-posterior selection
  rule* itself (Bayesian model averaging, an attack-targeted "uncertainty
  maximizing" selector, entropy-weighted selection, cross-round momentum
  correlation, literal Metropolis-Hastings); and (D) *structural/hybrid*
  defenses that combine CENSOR with the repo's other baselines (Soteria-style
  pruning, clipping, Top-k sparsification) or attack the value/index
  correspondence itself (output permutation).


## Kaggle / general setup

On Kaggle, turn on GPU if available (Settings -> Accelerator -> GPU T4 x2 or
better). Kaggle normally already ships PyTorch and torchvision, but the cell
below installs anything missing so the notebook also works in a fresh
environment. If this repo is uploaded as a Kaggle dataset, set `REPO_DIR` to
the folder containing `censor_frobenius_variations.py`.

**If you don't want to upload the companion `.py` file at all**, use
`censor_kaggle_standalone.ipynb` instead -- it is fully self-contained
(no external file needed) and reads CIFAR-10 directly from a Kaggle
input dataset.


In [ ]:
import importlib.util
import subprocess
import sys

def ensure(pkg, pip_name=None):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], check=True)

# Kaggle images already include these, but this keeps the notebook portable.
for pkg, pip_name in [("torch", "torch"), ("torchvision", "torchvision"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    ensure(pkg, pip_name)

print("Dependencies OK")

In [ ]:
from pathlib import Path
import os, sys

REPO_DIR = Path.cwd()
if not (REPO_DIR / 'censor_frobenius_variations.py').exists():
    candidates = list(Path('/kaggle/input').glob('**/censor_frobenius_variations.py')) if Path('/kaggle/input').exists() else []
    if candidates:
        REPO_DIR = candidates[0].parent

sys.path.insert(0, str(REPO_DIR))
print('Using repo:', REPO_DIR)

In [ ]:
import torch
import pandas as pd
from argparse import Namespace
from censor_frobenius_variations import (
    run_experiment, summarize, run_trials_ablation, run_layerwise_vs_entire_ablation,
    run_block_alignment_ablation, run_dirichlet_alpha_ablation, build_variant_registry,
)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('No GPU detected -- this still runs on CPU, just slower. On Kaggle, enable a GPU accelerator for the heavier run below.')

## Variations (32 total)

**Baseline / from the original repo extraction:**
- `raw_no_defense`: original private gradient, the leakage baseline (no defense at all).
- `repo_layerwise_as_is`: the *exact* arithmetic in the official `defense.py` (confirmed against GitHub), including its real orthogonality bug (divides by `||g||_F` instead of `||g||_F^2`).
- `layerwise_frobenius`: mathematically correct per-layer Frobenius projection using `<r,g> / ||g||_F^2` (paper Eq. 12, corrected).
- `global_frobenius`: exact projection after flattening the whole model gradient into one vector (paper's "Entire gradient" Table VII row).
- `final_layer_only_frobenius`: protects only the final FC layer.
- `row_wise_final_layer_frobenius`: **deliberate negative control** -- randomizes row *direction*, preserves row *norm* exactly. Should leak like `raw_no_defense`.
- `block_frobenius`: projects fixed-size contiguous chunks (`--block-size`), interpolating layerwise <-> global.
- `low_rank_structured_frobenius`: random candidate is rank-r (`--low-rank`) instead of full Gaussian.
- `noisy_global_frobenius`: `global_frobenius` + calibrated Gaussian noise (`--noise-sigma`), a DP-SGD-style hybrid.
- `cold_posterior_global_frobenius`: the paper's own trial-selection idea (deterministic argmin over `--trials`).
- `temperature_weighted_cold_posterior`: same trial pool, softmax-weighted choice (`--temperature`) instead of hard argmin.

**New, Axis A -- structure of the random candidate g_r:**
- `spectral_subspace_frobenius`: g_r drawn from the gradient's own top-k left-singular subspace (`--spectral-top-k`), not an unconstrained Gaussian.
- `dct_domain_frobenius`: g_r's coefficients are Gaussian *in a 2D DCT basis*, then inverse-transformed.
- `johnson_lindenstrauss_frobenius`: g_r constrained to a random subspace of dimension `--jl-dim-frac * numel` via a JL random projection.
- `rademacher_sign_frobenius`: g_r is elementwise +-1 (Rademacher) instead of Gaussian -- same measure-zero-hyperplane argument, different tail shape.
- `laplace_structured_frobenius`: g_r drawn from Laplace(0,1) -- the classical pure-DP noise distribution, heavier-tailed than Gaussian.

**New, Axis B -- what gets normalized, and how:**
- `spectral_norm_matched_frobenius`: Phase-2 rescale matches the *spectral* norm, not the Frobenius norm.
- `row_norm_equalized_frobenius`: **candidate real defense** -- forces every FC row to the same norm, directly zeroing the row-norm attack's information content.
- `row_norm_permuted_frobenius`: preserves the *multiset* of row norms but permutes which class gets which -- provably drops the row-norm attack to 1/C while leaving any aggregate statistic (and the untouched bias-sign channel) unchanged.
- `dirichlet_mass_reallocation_frobenius`: continuous knob (`--dirichlet-alpha`) between exact preservation (alpha->0) and exact equalization (alpha->inf) of row-norm mass.
- `bias_only_frobenius` / `weight_only_frobenius`: protect only the bias or only the weight, isolating which attack (sign-based vs norm-based) each half of the final layer is responsible for.

**New, Axis C -- the cold-posterior selection rule:**
- `bayesian_model_averaging_frobenius`: average the top-k lowest-loss trials (`--bma-top-k`) instead of a hard argmin -- a posterior-mean estimator instead of a MAP point estimate.
- `label_uncertainty_maximizing_frobenius`: selection rule that directly minimizes the row-norm attack's confidence margin instead of the training loss -- an explicit, attack-tuned "adversarial defense" included as a cautionary example (obfuscated-gradients-style risk), not a recommendation.
- `entropy_weighted_cold_posterior`: samples trials weighted by predictive entropy rather than negative loss.
- `momentum_correlated_cold_posterior`: biases selection toward the previous round's accepted gradient (`--momentum`) -- probes a *cross-round* correlation channel the single-round proxy can't otherwise see.
- `metropolis_hastings_orthogonal`: literal Metropolis-Hastings accept/reject Markov chain (the paper's own originally-proposed Obstacle-2 mechanism), instead of the T-iid-trials-then-argmax approximation the paper actually ships.

**New, Axis D -- structural / hybrid defenses:**
- `permuted_output_frobenius`: also secretly permutes the FC row/bias order -- a defense on the *label/index correspondence*, not the gradient values (with an explicit caveat about why this isn't practical).
- `class_grouped_block_frobenius`: projects each class's (weight row + bias) jointly as one block.
- `soteria_hybrid_frobenius`: per-layer magnitude pruning (`--prune-fraction`, Aji & Heafield / the paper's "Sparsi" baseline) + orthogonal projection.
- `clip_then_frobenius`: global-norm clipping (`--clip-bound`, Wei et al.'s DP clipping, paper Eq. 8) + orthogonal projection.
- `topk_sparsify_then_frobenius`: global Top-k sparsification (`--keep-fraction`, paper Eq. 9) + orthogonal projection.


In [ ]:
args = Namespace(
    dataset='CIFAR10',       # use CIFAR100 after the quick smoke test works
    data_path='./data',
    download=True,
    train_batches=300,       # increase to 1000+ for stronger model signals
    batch_size=64,
    eval_samples=64,         # increase to 200+ for more stable leak rates
    trials=20,               # CENSOR proposal count T; paper default is 20
    lr=0.03,
    update_lr=1e-4,
    block_size=256,          # block_frobenius granularity, in elements
    low_rank=4,              # rank for low_rank_structured_frobenius
    noise_sigma=0.1,         # relative Gaussian noise scale for noisy_global_frobenius
    temperature=0.05,        # softmax temperature for temperature_weighted_cold_posterior
    spectral_top_k=4,        # top-k singular directions for spectral_subspace_frobenius
    jl_dim_frac=0.1,         # random-subspace dimension fraction for johnson_lindenstrauss_frobenius
    dirichlet_alpha=1.0,     # concentration for dirichlet_mass_reallocation_frobenius
    bma_top_k=5,             # trials averaged in bayesian_model_averaging_frobenius
    momentum=0.5,            # momentum weight in momentum_correlated_cold_posterior
    mh_temperature=0.01,     # Boltzmann temperature for metropolis_hastings_orthogonal
    prune_fraction=0.5,      # fraction zeroed in soteria_hybrid_frobenius
    clip_bound=1.0,          # global Frobenius clip bound for clip_then_frobenius
    keep_fraction=0.5,       # fraction kept in topk_sparsify_then_frobenius
    seed=1234,
    cpu=False,
    out='censor_variation_results.csv',
)

# FIXED (see module docstring): run_experiment now consistently returns
# (rows, meta), and summarize takes both.
rows, meta = run_experiment(args)
summarize(rows, meta)
df = pd.DataFrame(rows)
df.to_csv(args.out, index=False)
df.head()

In [ ]:
summary = (
    df.groupby('name')
      .agg(samples=('sample', 'count'), leaks_norm=('leaked_norm', 'sum'), leaks_sign=('leaked_sign', 'sum'),
           avg_abs_cosine=('cosine', lambda s: s.abs().mean()), avg_norm_ratio=('norm_ratio', 'mean'))
      .reset_index()
)
summary['leak_rate_norm_pct'] = 100 * summary['leaks_norm'] / summary['samples']
summary['leak_rate_sign_pct'] = 100 * summary['leaks_sign'] / summary['samples']
summary.sort_values('leak_rate_norm_pct', ascending=False)

In [ ]:
import matplotlib.pyplot as plt

plot_df = summary.sort_values('leak_rate_norm_pct')
plt.figure(figsize=(10, 11))
plt.barh(plot_df['name'], plot_df['leak_rate_norm_pct'], color='#4C72B0', label='row-norm proxy')
plt.barh(plot_df['name'], plot_df['leak_rate_sign_pct'], color='#DD8452', alpha=0.55, label='bias-sign proxy (paper Eq. 7)')
plt.xlabel('Label inference leak rate (%)')
plt.ylabel('Variation')
plt.title('Attack leads against 32 CENSOR-style Frobenius variations\n(NOT the paper\'s reconstruction metrics -- see notebook intro)')
plt.legend(loc='lower right')
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

## Ablation 1: Layer-wise vs. Entire gradient (mirrors paper Table VII)

The paper reports layer-wise projection slightly outperforms whole-model
("entire gradient") projection. We check the same comparison on our proxy.


In [ ]:
layer_rows = run_layerwise_vs_entire_ablation(args)
layer_df = pd.DataFrame(layer_rows)
layer_df

## Ablation 2: Number of cold-posterior trials T (mirrors paper Figure 9)

The paper sweeps T in {0,10,20,30,40,50} and finds diminishing returns past
T=20 on GIAS/ImageNet reconstruction metrics. We sweep the same values on
our proxy against `cold_posterior_global_frobenius`.


In [ ]:
trials_rows = run_trials_ablation(args, trial_values=[1, 5, 10, 20, 30, 40, 50])
trials_df = pd.DataFrame(trials_rows)
trials_df

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(trials_df['trials'], trials_df['leak_rate_norm_pct'], marker='o', label='row-norm proxy')
plt.plot(trials_df['trials'], trials_df['leak_rate_sign_pct'], marker='s', label='bias-sign proxy')
plt.xlabel('Number of cold-posterior trials T')
plt.ylabel('Leak rate (%)')
plt.title('cold_posterior_global_frobenius: leak rate vs. T\n(paper\'s Fig. 9 analogue, on our proxy metric)')
plt.legend()
plt.tight_layout()
plt.show()

## Ablation 3: `block_frobenius` alignment loophole

`block_size` values that evenly divide the FC layer's `in_features` (128 for
`TinyCIFARNet`) can silently reconstruct the exact per-row norm from
per-block norms -- the same loophole `row_wise_final_layer_frobenius`
exploits on purpose, here by accident. Compare aligned (32, 64, 128, 256)
vs. non-aligned (96, 300) block sizes.


In [ ]:
block_rows = run_block_alignment_ablation(args, block_sizes=[32, 64, 96, 128, 256, 300])
block_df = pd.DataFrame(block_rows)
block_df

## Ablation 4 (new, no paper analogue): Dirichlet mass-reallocation sweep

`dirichlet_mass_reallocation_frobenius(alpha)` continuously interpolates
between exact row-norm preservation (alpha -> 0, expect leak like
`row_wise_final_layer_frobenius`) and exact equalization (alpha -> inf,
expect leak like `row_norm_equalized_frobenius`). This visualizes the
row-norm leak channel as a *continuous* function of a single knob, rather
than only at its two limiting endpoints.


In [ ]:
dirichlet_rows = run_dirichlet_alpha_ablation(args, alpha_values=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0])
dirichlet_df = pd.DataFrame(dirichlet_rows)

plt.figure(figsize=(7, 4.5))
plt.semilogx(dirichlet_df['alpha'], dirichlet_df['leak_rate_norm_pct'], marker='o')
plt.xlabel('Dirichlet concentration alpha (log scale)')
plt.ylabel('Row-norm proxy leak rate (%)')
plt.title('dirichlet_mass_reallocation_frobenius: leak rate vs. alpha')
plt.tight_layout()
plt.show()
dirichlet_df

## Reading the results

A few things worth checking once the run finishes:

- `row_wise_final_layer_frobenius` should leak close to `raw_no_defense`'s rate on **both** proxies -- if it doesn't, that's a sign something in the harness (not the defense) changed between variants.
- `row_norm_permuted_frobenius` should defeat the row-norm proxy (leak ~1/num_classes) while `leaked_sign` stays high, since the bias is untouched -- a clean demonstration that these two proxies are separable, non-redundant channels.
- `bias_only_frobenius` should defeat the sign proxy but leave the norm proxy at its raw rate; `weight_only_frobenius` should do the opposite. Together they should roughly explain `final_layer_only_frobenius`'s combined effect.
- **`block_frobenius` can leak by accident**, not just by design: compare `--block-size 64/128/256` (all divide `in_features=128`) against `96/300` (non-aligned) in Ablation 3.
- Compare `layerwise_frobenius` vs `global_frobenius` (Ablation 1) to see the paper's own Table VII effect reproduced on our proxy.
- Compare `cold_posterior_global_frobenius` against `temperature_weighted_cold_posterior`, `bayesian_model_averaging_frobenius`, and `metropolis_hastings_orthogonal` at a few different trial counts/temperatures -- if the deterministic argmin leaks noticeably more than the softer selection rules at low sample counts, that is evidence the selection rule itself carries a little information; if not, it supports the paper's implicit claim that leak-resistance comes entirely from the *proposal* distribution's orthogonality, not the selection rule.
- `noisy_global_frobenius`'s `avg_norm_ratio` will be `>> 1` since noise is added without renormalizing (by design, to mimic a DP mechanism) -- don't compare its norm_ratio directly against the other variants without accounting for that.
- `label_uncertainty_maximizing_frobenius` is *designed* to look artificially strong on exactly this proxy metric -- that is the point of including it (see its docstring): it is a cautionary example of overfitting a defense to one specific attack statistic, not a defense you should actually deploy.

**For the full mathematical treatment of every variant, and a detailed
discussion of "is a 100% leak rate here a break of CENSOR?", see the
accompanying PDF research report.**


## Heavier Run

After the quick run works, try CIFAR100 and more samples, and sweep the new hyperparameters:

```python
args.dataset = 'CIFAR100'
args.train_batches = 1000
args.eval_samples = 200
args.trials = 50
rows, meta = run_experiment(args)
summarize(rows, meta)
```

```python
# Sweep block granularity -- note 32/128/512 all evenly divide the FC
# layer's row length (128) and may trigger the row-alignment leak documented
# above; 96/300/700 are deliberately non-aligned for comparison.
for bs in [32, 96, 128, 300, 700, 2048]:
    args.block_size = bs
    rows, meta = run_experiment(args)
    summarize(rows, meta)
```

For command-line use, run:

```bash
python censor_frobenius_variations.py --dataset CIFAR10 --download --eval-samples 64 --trials 20 --run-ablations
python censor_frobenius_variations.py --dataset CIFAR100 --download --train-batches 1000 --eval-samples 200 --trials 50 \
    --block-size 512 --low-rank 8 --noise-sigma 0.2 --temperature 0.1 --dirichlet-alpha 5 --run-ablations
```
